In [ ]:
import re
import shutil
import pandas as pd
from pathlib import Path

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"c:\Users\Usuario\Documents\Monroe County ORRI")
DATA_FILE  = BASE_DIR / "Monroe County ORRI \u2014 Project Log FLATRIVER (1).xlsx"
COMBINED   = BASE_DIR / "agreementsExtractedTables" / "All_Agreements_Combined.xlsx"
AGREEMENTS = BASE_DIR / "agreementsExtractedTables" / "Excel files man"
ULTIMATE   = BASE_DIR / "UltimateFiles"
BOOKPAGE   = BASE_DIR / "UTIC files (book-page)"
DEST       = BASE_DIR / "Deal Files (by Agreement)"

print(f"Data file  : {DATA_FILE.name}")
print(f"Combined   : {COMBINED.name}")
print(f"UltimateFiles exists : {ULTIMATE.exists()}")
print(f"Book-Page  exists    : {BOOKPAGE.exists()}")
print(f"Output     : {DEST}")

In [ ]:
# ── Load DB FINDING: Col A (Heritage Lease Name) + Col X (DEAL NAME) + Col D (Book/Page) ──
df_db = pd.read_excel(DATA_FILE, sheet_name="DB FINDING")

id_col   = df_db.columns[0]   # Heritage Lease Name
bp_col   = df_db.columns[3]   # Book/Page
deal_col = df_db.columns[23]  # DEAL NAME

print(f"DB FINDING columns used:")
print(f"  A ({id_col}), D ({bp_col}), X ({deal_col})")
print(f"  Total rows: {len(df_db)}")

# Deduplicate by Heritage Lease Name → one DEAL NAME per lease
df_dedup = df_db[[id_col, bp_col, deal_col]].dropna(subset=[id_col]).drop_duplicates(subset=[id_col], keep="first").copy()
df_dedup[id_col]   = df_dedup[id_col].astype(str).str.strip()
df_dedup[deal_col] = df_dedup[deal_col].astype(str).str.strip()

# lease_to_deal: Heritage Lease Name → DEAL NAME (fall back to lease name if blank)
lease_to_deal = {}
for _, row in df_dedup.iterrows():
    deal = row[deal_col]
    if not deal or deal.lower() in ("nan", ""):
        deal = row[id_col]  # fallback
    lease_to_deal[row[id_col]] = deal

# lease_to_bp: Heritage Lease Name → normalised "book-page" string
def norm_bp(val):
    if pd.isna(val): return None
    s = re.sub(r'^(OR|LR)\s+', '', str(val).strip(), flags=re.IGNORECASE)
    parts = re.split(r'[/\-]', s)
    if len(parts) == 2:
        try: return f"{int(parts[0].strip())}-{int(parts[1].strip())}"
        except: pass
    return None

lease_to_bp = {}
for _, row in df_dedup.iterrows():
    bp = norm_bp(row[bp_col])
    if bp:
        lease_to_bp[row[id_col]] = bp

print(f"\nUnique leases with DEAL NAME : {len(lease_to_deal)}")
print(f"Unique leases with Book/Page : {len(lease_to_bp)}")

In [ ]:
# ── Load All_Agreements_Combined → agreement → [lease numbers] ─────────────
df_comb = pd.read_excel(COMBINED)
df_comb["Lease No."]   = df_comb["Lease No."].astype(str).str.strip()
df_comb["Source File"] = df_comb["Source File"].astype(str).str.strip()

agreement_to_leases = (
    df_comb[df_comb["Lease No."].str.upper() != "NAN"]
    .groupby("Source File")["Lease No."]
    .apply(lambda x: list(x.unique()))
    .to_dict()
)

# All 17 agreement names from the Excel files man folder
agreement_names = sorted([x.stem for x in AGREEMENTS.glob("*.xlsx") if not x.name.startswith("~$")])

print(f"Agreement files found: {len(agreement_names)}")
for a in agreement_names:
    n = len(agreement_to_leases.get(a, []))
    print(f"  {a:55s} {n:>3d} leases")

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

def sanitize_name(name):
    """Remove characters that are invalid in Windows folder names."""
    name = re.sub(r'[<>:"/\\|?*]', '-', str(name))
    name = name.strip('. ')
    return name or "_unnamed"

def find_in_ultimate(lease_id):
    """
    Return the primary UltimateFiles folder for a lease.
    Prefers exact match WITHOUT the ___HG_ORRI suffix.
    """
    primary = ULTIMATE / lease_id
    if primary.exists() and primary.is_dir():
        return primary
    # Accept suffixed variant as fallback
    matches = [p for p in ULTIMATE.iterdir()
               if p.is_dir() and p.name.startswith(lease_id)]
    return matches[0] if matches else None

def find_in_bookpage(bp_str):
    """Return folder in UTIC files (book-page) matching 'book-page' string."""
    if not bp_str:
        return None
    target = BOOKPAGE / bp_str
    if target.exists() and target.is_dir():
        return target
    return None

def safe_dest(parent: Path, name: str) -> Path:
    """Return a non-colliding destination path, appending _2, _3 … as needed."""
    candidate = parent / name
    if not candidate.exists():
        return candidate
    counter = 2
    while True:
        candidate = parent / f"{name}_{counter}"
        if not candidate.exists():
            return candidate
        counter += 1

print("Helper functions defined.")

In [ ]:
# ── Main copy loop: match by Heritage Lease Name ───────────────────────────
if DEST.exists():
    print(f"Removing existing destination: {DEST}")
    shutil.rmtree(DEST)
DEST.mkdir()

unmatched = []   # (agreement, lease_id, deal_name) — no UltimateFiles folder found
copied    = 0

for agreement in agreement_names:
    subfolder = DEST / sanitize_name(agreement)
    subfolder.mkdir(exist_ok=True)

    leases = agreement_to_leases.get(agreement, [])
    if not leases:
        print(f"  [!!]  {agreement} — no leases in combined table")
        continue

    for lease_id in leases:
        deal_name = lease_to_deal.get(lease_id, lease_id)
        src = find_in_ultimate(lease_id)

        if src:
            dest_path = safe_dest(subfolder, sanitize_name(deal_name))
            shutil.copytree(src, dest_path)
            print(f"  [OK]  [{agreement[:30]}] {lease_id} -> {dest_path.name}")
            copied += 1
        else:
            unmatched.append({"agreement": agreement, "lease_id": lease_id, "deal_name": deal_name})

print(f"\nPrimary pass complete.")
print(f"  Copied   : {copied}")
print(f"  Unmatched: {len(unmatched)}")

In [ ]:
# ── Fallback: match by Book/Page ───────────────────────────────────────────
final_missing = []
fallback_copied = 0

for entry in unmatched:
    agreement = entry["agreement"]
    lease_id  = entry["lease_id"]
    deal_name = entry["deal_name"]

    bp  = lease_to_bp.get(lease_id)
    src = find_in_bookpage(bp)

    if src:
        subfolder = DEST / sanitize_name(agreement)
        dest_path = safe_dest(subfolder, sanitize_name(deal_name))
        shutil.copytree(src, dest_path)
        print(f"  [BP]  [{agreement[:30]}] {lease_id} (bp={bp}) -> {dest_path.name}")
        fallback_copied += 1
    else:
        final_missing.append(entry)
        bp_note = bp if bp else "no book/page"
        print(f"  [--]  [{agreement[:30]}] {lease_id} ({bp_note}) — not found")

print(f"\nFallback pass complete.")
print(f"  Copied via book/page : {fallback_copied}")
print(f"  Still missing        : {len(final_missing)}")

In [ ]:
# ── Final Report ───────────────────────────────────────────────────────────
total_copied = copied + fallback_copied

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Agreement subfolders created : {len(agreement_names)}")
print(f"Folders copied (by lease ID) : {copied}")
print(f"Folders copied (by book/page): {fallback_copied}")
print(f"Total copied                 : {total_copied}")
print(f"Final missing                : {len(final_missing)}")
print()

if final_missing:
    df_missing = pd.DataFrame(final_missing)
    print("Unresolved entries:")
    display(df_missing)
else:
    print("All leases resolved.")